# Local, Free Search Relevance Pipeline for Dry Bulk Maritime News

This notebook builds a **hybrid search engine** over the scraped article corpus
(Breakwave, Hellenic Shipping News, Splash247) entirely with **free, local,
open-source models** — no paid APIs, no internet calls after the models are
downloaded once from Hugging Face.

Pipeline, end to end:

1. **Load & clean** the three CSVs into one article table.
2. **Chunk** long articles so they fit the embedding model's context window.
3. **BM25** (sparse / keyword) retrieval — fast candidate generation.
4. **Dense** retrieval with a small sentence-embedding model + FAISS — catches
   semantic matches BM25 misses (synonyms, paraphrases).
5. **Hybrid fusion** of BM25 + dense via Reciprocal Rank Fusion (RRF).
6. **Cross-encoder reranking** of the top candidates for precision.
7. Optional **recency boost** — shipping news goes stale fast.
8. **Offline evaluation** (automatic known-item proxy + a manual query set)
   comparing every stage on quality *and* speed.
9. A **search interface** cell for everyday use.

This mirrors a production retrieval-augmented-search stack (think: the
retrieval half of RAG) at a scale (~50k articles) that runs comfortably on a
laptop CPU.

In [1]:
# One-line install (uncomment to run). Everything below is free / open-source
# and runs locally on CPU — no OpenAI/Anthropic keys required.
# !pip install -U sentence-transformers rank_bm25 faiss-cpu numpy pandas tqdm regex

import json
import random
import re
import string
import time
from dataclasses import dataclass
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm.auto import tqdm

# ---- Reproducibility -------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---- Paths (all relative to this notebook's folder, no hardcoded absolute
#      paths so this runs on any machine) -----------------------------------
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "exported_csv"
CACHE_DIR = BASE_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)

# ---- Rebuild flags ----------------------------------------------------------
# Flip to True to force recomputation even if a cache file already exists on
# disk (e.g. after changing the embedding model or the chunking parameters).
FORCE_REBUILD_PROD_INDEX = False
FORCE_REBUILD_EVAL_INDEX = False

pd.set_option("display.max_colwidth", 80)
print("Setup complete. Data dir:", DATA_DIR.resolve())

Setup complete. Data dir: D:\VS_code\Dry_Bulk_maritime_research\exported_csv


## Stage 0 — Load & clean the three sources

**Why this matters:** the three scrapers produced three different schemas and
three different date formats. If we don't normalize this up front, every
downstream stage (chunking, dates for the recency boost, dedup) breaks
silently. We:

1. Rename `extracted_articles_new.csv` (Breakwave) onto the common schema
   (`source, title, date, author, link, text`) used by the Hellenic and
   Splash exports.
2. Parse each source's date format explicitly rather than letting pandas
   guess (guessing is how `30/06/2026` silently becomes June 30 in one file
   and a parse failure — or worse, the wrong date — in another).
3. Drop exact duplicate links, then catch near-duplicate titles (e.g. the
   same piece picked up by two scrapers with trailing whitespace/punctuation
   differences).
4. Flag/drop empty or near-empty article bodies — an article with no text
   can't be meaningfully embedded or keyword-matched.

In [2]:
def load_sources(data_dir: Path) -> pd.DataFrame:
    """Load the three scraped CSVs and normalize them onto one schema.

    Returns a single DataFrame with columns:
    source, title, date (raw string), author, link, text
    """
    hellenic = pd.read_csv(data_dir / "Hellenic_df_full_with_articles.csv")
    splash = pd.read_csv(data_dir / "splash_df_full_with_articles.csv")

    breakwave = pd.read_csv(data_dir / "extracted_articles_new.csv")
    breakwave = breakwave.rename(
        columns={"Link": "link", "Title": "title", "Content": "text"}
    )
    breakwave["source"] = "Breakwave"
    breakwave["author"] = None
    # Breakwave's own "Datetime" column is already ISO (YYYY-MM-DD); keep the
    # raw "Date" string too so every source goes through the same date-parsing
    # code path in the next stage.
    breakwave["date"] = breakwave["Date"]
    breakwave = breakwave[["source", "title", "date", "author", "link", "text"]]

    common_cols = ["source", "title", "date", "author", "link", "text"]
    df = pd.concat(
        [hellenic[common_cols], splash[common_cols], breakwave[common_cols]],
        ignore_index=True,
    )
    return df


articles_raw = load_sources(DATA_DIR)
print("Loaded", len(articles_raw), "rows")
print(articles_raw["source"].value_counts())

Loaded 50373 rows
source
splash       47420
Breakwave     2612
Hellenic       341
Name: count, dtype: int64


In [3]:
def parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    """Parse each source's date string with its own known format.

    - Breakwave: "February 13, 2026"            -> month name, day-first=False
    - Hellenic:  "30/06/2026"                    -> day/month/year (day-first)
    - Splash:    "April 01, 2014"                -> month name, day-first=False

    Any date that still fails to parse becomes NaT and is counted, rather than
    silently dropped, so data-quality issues stay visible.
    """
    df = df.copy()
    parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")

    format_by_source = {
        "Hellenic": {"dayfirst": True},
        "Breakwave": {"dayfirst": False},
        "splash": {"dayfirst": False},
    }

    for source, kwargs in format_by_source.items():
        mask = df["source"] == source
        parsed.loc[mask] = pd.to_datetime(df.loc[mask, "date"], errors="coerce", **kwargs)

    df["parsed_date"] = parsed

    n_failed = df["parsed_date"].isna().sum()
    print(f"Date parsing: {n_failed} / {len(df)} rows failed to parse ({n_failed / len(df):.2%})")
    if n_failed:
        print("Failures by source:")
        print(df.loc[df["parsed_date"].isna(), "source"].value_counts())
    return df


articles_raw = parse_dates(articles_raw)

Date parsing: 0 / 50373 rows failed to parse (0.00%)


In [4]:
def normalize_title(title: str) -> str:
    """Lowercase, strip punctuation and collapse whitespace for comparison."""
    title = str(title).lower().strip()
    title = title.translate(str.maketrans("", "", string.punctuation))
    title = re.sub(r"\s+", " ", title)
    return title


def dedupe_articles(df: pd.DataFrame, title_sim_threshold: float = 0.92) -> pd.DataFrame:
    """Remove duplicate articles.

    Two passes:
    1. Exact-link duplicates (cheap, catches the same URL scraped twice).
    2. Near-duplicate titles: block by the first 6 normalized words (so we
       only ever compare titles that already look alike) and flag pairs
       whose difflib similarity ratio exceeds `title_sim_threshold`. This is
       O(n) blocks x small-block-pairwise rather than O(n^2) over the whole
       corpus, so it stays cheap even at ~50k rows. We keep the first
       occurrence of each duplicate group.
    """
    df = df.copy()
    n0 = len(df)

    df = df.drop_duplicates(subset="link", keep="first")
    n1 = len(df)
    print(f"Dropped {n0 - n1} exact-link duplicates")

    df["_norm_title"] = df["title"].map(normalize_title)
    df["_block_key"] = df["_norm_title"].str.split().str[:6].str.join(" ")

    keep_mask = np.ones(len(df), dtype=bool)
    df = df.reset_index(drop=True)

    for _, idxs in df.groupby("_block_key").groups.items():
        idxs = list(idxs)
        if len(idxs) < 2:
            continue
        for i in range(len(idxs)):
            if not keep_mask[idxs[i]]:
                continue
            for j in range(i + 1, len(idxs)):
                if not keep_mask[idxs[j]]:
                    continue
                ratio = SequenceMatcher(
                    None, df.loc[idxs[i], "_norm_title"], df.loc[idxs[j], "_norm_title"]
                ).ratio()
                if ratio >= title_sim_threshold:
                    keep_mask[idxs[j]] = False  # drop the later duplicate

    n_near_dupes = (~keep_mask).sum()
    df = df[keep_mask].drop(columns=["_norm_title", "_block_key"]).reset_index(drop=True)
    print(f"Dropped {n_near_dupes} near-duplicate-title articles")
    print(f"Remaining: {len(df)} / {n0} original rows")
    return df


articles_raw = dedupe_articles(articles_raw)

Dropped 0 exact-link duplicates
Dropped 1303 near-duplicate-title articles
Remaining: 49070 / 50373 original rows


In [5]:
def flag_short_text(df: pd.DataFrame, min_chars: int = 50) -> pd.DataFrame:
    """Drop rows whose article body is empty or too short to be searchable.

    A short/empty body can't be meaningfully chunked, embedded, or
    keyword-matched, and would just add noise (or crash the chunker) later.
    """
    df = df.copy()
    df["text"] = df["text"].fillna("")
    text_len = df["text"].str.len()

    n_empty = (text_len == 0).sum()
    n_short = ((text_len > 0) & (text_len < min_chars)).sum()
    print(f"Empty text: {n_empty} rows | Short text (<{min_chars} chars, non-empty): {n_short} rows")
    print("By source:")
    print(df.loc[text_len < min_chars, "source"].value_counts())

    kept = df[text_len >= min_chars].reset_index(drop=True)
    print(f"Dropped {len(df) - len(kept)} rows total; {len(kept)} remain")
    return kept


articles = flag_short_text(articles_raw, min_chars=50)
articles = articles.rename(columns={"parsed_date": "date_parsed"})
articles["article_id"] = articles.index
articles["text_len"] = articles["text"].str.len()

Empty text: 214 rows | Short text (<50 chars, non-empty): 0 rows
By source:
source
Hellenic    203
splash       11
Name: count, dtype: int64
Dropped 214 rows total; 48856 remain


In [6]:
def summarize(df: pd.DataFrame) -> None:
    """Print rows-per-source, date range, and text-length distribution."""
    print("=== Rows per source ===")
    print(df["source"].value_counts())
    print()
    print("=== Date range (parsed) ===")
    print("min:", df["date_parsed"].min(), "| max:", df["date_parsed"].max())
    print("Unparseable dates remaining:", df["date_parsed"].isna().sum())
    print()
    print("=== Text length (characters) distribution ===")
    print(df["text_len"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))


summarize(articles)
articles.head(3)

=== Rows per source ===
source
splash       47245
Breakwave     1474
Hellenic       137
Name: count, dtype: int64

=== Date range (parsed) ===
min: 2012-11-05 00:00:00 | max: 2026-09-09 00:00:00
Unparseable dates remaining: 0

=== Text length (characters) distribution ===
count    48856.000000
mean      1570.205461
std       1407.574388
min        239.000000
10%        596.000000
25%        802.000000
50%       1150.000000
75%       1739.000000
90%       3059.000000
99%       7053.450000
max      67608.000000
Name: text_len, dtype: float64


,source,title,date,author,link,text,date_parsed,article_id,text_len
0,Hellenic,Baltic Dry Index climbs to 2501 up 11 points,30/06/2026,NaN,https://www.hellenicshippingnews.com/baltic-dry-index-climbs-to-2501-up-11-p...,Baltic Dry Index climbs to 2501 up 11 points in Dry Bulk Market 30/06/2026 T...,2026-06-30,0,644
1,Hellenic,India’s thermal coal imports drop to 4-year low on local output,19/06/2026,NaN,https://www.hellenicshippingnews.com/indias-thermal-coal-imports-drop-to-4-y...,India’s thermal coal imports drop to 4-year low on local output in Dry Bulk ...,2026-06-19,1,1606
2,Hellenic,Baltic Dry Index falls to 2729 down 42 points,11/06/2026,NaN,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2729-down-42-...,Baltic Dry Index falls to 2729 down 42 points in Dry Bulk Market 11/06/2026 ...,2026-06-11,2,651


## Stage 1 — Chunking: why and how

Embedding models have a fixed context window (bge-small-en-v1.5 truncates at
512 tokens, roughly 350-400 words). Our articles average ~260 words but a long
tail runs past 4,000 words — if we embed the whole article as one vector, the
model silently truncates it and everything after the truncation point is
invisible to search.

**Fix:** split each article's `title + text` into overlapping ~200-word
chunks (50-word overlap), embed every chunk separately, then **aggregate
chunk scores up to the article level by taking the max score across its
chunks**. The overlap means a key sentence that falls near a chunk boundary
still appears whole in at least one chunk. Max-aggregation (rather than mean)
is deliberate: we want "does *any part* of this article match the query"
not "does the *average* of this article match" — a single highly relevant
paragraph in an otherwise unrelated long article should still surface the
article.

In [7]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> list[str]:
    """Split `text` into overlapping word-count chunks.

    Step size = chunk_size - overlap. The last chunk is kept even if shorter
    than chunk_size (we never drop the tail of an article).
    """
    words = text.split()
    if len(words) <= chunk_size:
        return [text]

    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            break
        chunks.append(" ".join(chunk_words))
        if start + chunk_size >= len(words):
            break
    return chunks


def build_chunks(df: pd.DataFrame, text_field: str) -> pd.DataFrame:
    """Chunk every article's `text_field` and return a long (article_id, chunk)
    table. `text_field` is a precomputed column name (e.g. "search_text").
    """
    records = []
    for article_id, text in zip(df["article_id"], df[text_field]):
        for chunk in chunk_text(text):
            records.append({"article_id": article_id, "chunk_text": chunk})
    chunks_df = pd.DataFrame(records)
    chunks_df["chunk_id"] = chunks_df.index
    return chunks_df


# Production search field: title + body, so a query matching only the title
# (e.g. "Baltic Dry Index falls") still retrieves the article.
articles["search_text"] = articles["title"].fillna("") + ". " + articles["text"].fillna("")

prod_chunks = build_chunks(articles, "search_text")
print(f"{len(articles)} articles -> {len(prod_chunks)} chunks "
      f"({len(prod_chunks) / len(articles):.2f} chunks/article on average)")

48856 articles -> 92290 chunks (1.89 chunks/article on average)


## Stage 2 — Sparse retrieval (BM25)

BM25 is a **keyword-matching** algorithm: it scores a document highly if it
contains the query's exact words (weighted by how rare those words are across
the corpus, and normalized for document length). It's fast, needs no model
download, and is extremely good at exact terms — ship names, index numbers,
company names — that a dense embedding model can sometimes blur together.
Its weakness is synonyms/paraphrases ("vessel" vs "ship"), which is exactly
what Stage 3 (dense retrieval) is for.

We tokenize with simple lowercasing + regex word-splitting + stopword
removal (no heavy NLP dependency needed for this).

In [8]:
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "then", "else", "for", "to",
    "of", "in", "on", "at", "by", "with", "from", "as", "is", "are", "was",
    "were", "be", "been", "being", "it", "its", "this", "that", "these",
    "those", "i", "you", "he", "she", "we", "they", "them", "his", "her",
    "their", "our", "your", "not", "no", "so", "than", "too", "very", "can",
    "will", "would", "could", "should", "has", "have", "had", "do", "does",
    "did", "about", "into", "up", "down", "out", "over", "under", "again",
    "further", "all", "any", "both", "each", "few", "more", "most", "other",
    "some", "such", "only", "own", "same",
}


def tokenize(text: str) -> list[str]:
    """Lowercase, extract word tokens, drop stopwords. Used for BM25."""
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if w not in STOPWORDS]


def build_bm25_index(chunks_df: pd.DataFrame) -> BM25Okapi:
    tokenized = [tokenize(t) for t in tqdm(chunks_df["chunk_text"], desc="Tokenizing for BM25")]
    return BM25Okapi(tokenized)


def bm25_search_chunks(bm25: BM25Okapi, chunks_df: pd.DataFrame, query: str) -> pd.Series:
    """Return a Series of BM25 scores indexed by chunk position."""
    scores = bm25.get_scores(tokenize(query))
    return pd.Series(scores, index=chunks_df.index)


def aggregate_chunk_scores_to_articles(
    chunk_scores: pd.Series, chunks_df: pd.DataFrame, top_k: int, allowed_mask: np.ndarray | None = None
) -> list[tuple[int, float]]:
    """Max-aggregate chunk-level scores to article-level, return top_k
    (article_id, score) pairs sorted by score descending.

    `allowed_mask` (same length/order as `chunks_df`) restricts aggregation to
    a subset of chunks -- e.g. only chunks from articles published in a given
    date window -- before taking the max per article. See `recent_window()`.
    """
    tmp = pd.DataFrame({"article_id": chunks_df["article_id"], "score": chunk_scores})
    if allowed_mask is not None:
        tmp = tmp[allowed_mask]
    article_scores = tmp.groupby("article_id")["score"].max()
    top = article_scores.sort_values(ascending=False).head(top_k)
    return list(top.items())


def bm25_search(
    bm25: BM25Okapi, chunks_df: pd.DataFrame, query: str, top_k: int = 200,
    allowed_mask: np.ndarray | None = None,
):
    chunk_scores = bm25_search_chunks(bm25, chunks_df, query)
    return aggregate_chunk_scores_to_articles(chunk_scores, chunks_df, top_k, allowed_mask=allowed_mask)


prod_bm25 = build_bm25_index(prod_chunks)
print(bm25_search(prod_bm25, prod_chunks, "baltic dry index falls", top_k=5))

Tokenizing for BM25:   0%|          | 0/92290 [00:00<?, ?it/s]

[(35, 28.730727037980568), (11, 28.730727037980568), (48, 28.730727037980568), (2, 28.730727037980568), (10, 28.730727037980568)]


## Stage 3 — Dense retrieval (embeddings + FAISS)

We embed every chunk with **`BAAI/bge-small-en-v1.5`** — a small (33M
parameter), free, CPU-friendly sentence embedding model that ranks well on
retrieval benchmarks for its size. BGE models recommend prefixing *queries*
(not documents) with an instruction string so the model knows it's encoding a
search query rather than a passage: `"Represent this sentence for searching
relevant passages: "`.

Vectors are L2-normalized so that **inner product = cosine similarity**, and
stored in a FAISS `IndexFlatIP` — an exact brute-force search. At ~50k
articles / ~100k chunks this is still fast (a few milliseconds per query) and
simpler/more accurate than an approximate index. If the corpus grew into the
millions of chunks, exact search would get slow and you'd switch to FAISS
`IndexHNSWFlat` (approximate nearest neighbor via a navigable small-world
graph) — faster at query time at the cost of a small, tunable recall loss.

Embeddings and the FAISS index are **cached to disk** (`cache/`) so re-running
this notebook doesn't re-embed ~100k chunks every time; set
`FORCE_REBUILD_PROD_INDEX = True` above to recompute after changing the model
or chunking parameters.

In [ ]:
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

_embed_model = None


def get_embed_model() -> SentenceTransformer:
    global _embed_model
    if _embed_model is None:
        _embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    return _embed_model


def embed_texts(texts: list[str], is_query: bool = False, batch_size: int = 64) -> np.ndarray:
    """Embed a list of texts with bge-small-en-v1.5, L2-normalized.

    `is_query=True` applies BGE's recommended query instruction prefix.
    Documents/chunks are embedded with no prefix, per the model card.
    """
    model = get_embed_model()
    if is_query:
        texts = [BGE_QUERY_PREFIX + t for t in texts]
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return embeddings.astype("float32")


def build_or_load_dense_index(
    chunks_df: pd.DataFrame, cache_prefix: str, force_rebuild: bool = False
):
    """Embed `chunks_df['chunk_text']` (with caching) and build a FAISS
    IndexFlatIP over the resulting normalized vectors.

    Returns (faiss_index, embeddings). Cache files:
      cache/{cache_prefix}_embeddings.npy
      cache/{cache_prefix}_faiss.index
    """
    emb_path = CACHE_DIR / f"{cache_prefix}_embeddings.npy"
    index_path = CACHE_DIR / f"{cache_prefix}_faiss.index"

    if not force_rebuild and emb_path.exists() and index_path.exists():
        print(f"Loading cached embeddings/index for '{cache_prefix}' from disk")
        embeddings = np.load(emb_path)
        index = faiss.read_index(str(index_path))
        return index, embeddings

    print(f"Embedding {len(chunks_df)} chunks for '{cache_prefix}' (this is the slow, one-time step)")
    embeddings = embed_texts(chunks_df["chunk_text"].tolist(), is_query=False)

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    np.save(emb_path, embeddings)
    faiss.write_index(index, str(index_path))
    print(f"Cached embeddings to {emb_path.name} and index to {index_path.name}")
    return index, embeddings


def _faiss_search(index: faiss.Index, query_vec: np.ndarray, k: int, allowed_positions: np.ndarray | None):
    """Search the flat index, optionally restricted to the given row positions
    (e.g. only chunks from articles in a recent date window). Returns
    (scores, positions) with FAISS's -1 padding dropped."""
    if allowed_positions is None:
        scores, positions = index.search(query_vec, min(k, index.ntotal))
    else:
        selector = faiss.IDSelectorBatch(allowed_positions.astype("int64"))
        params = faiss.SearchParameters(sel=selector)
        scores, positions = index.search(query_vec, min(k, len(allowed_positions)), params=params)
    keep = positions[0] >= 0
    return scores[0][keep], positions[0][keep]


def dense_search(
    index: faiss.Index, chunks_df: pd.DataFrame, query: str, top_k: int = 200, search_k_chunks: int = 2000,
    allowed_positions: np.ndarray | None = None,
):
    """Embed `query`, search the FAISS index, max-aggregate chunk hits to
    article level, return top_k (article_id, score) pairs.

    `allowed_positions` restricts the search to a subset of row positions in
    `chunks_df` (see `recent_window()`), so recency-filtered queries never
    see chunks outside the requested date window in the first place.
    """
    query_vec = embed_texts([query], is_query=True)
    scores, positions = _faiss_search(index, query_vec, search_k_chunks, allowed_positions)

    tmp = pd.DataFrame(
        {
            "article_id": chunks_df.iloc[positions]["article_id"].values,
            "score": scores,
        }
    )
    article_scores = tmp.groupby("article_id")["score"].max()
    top = article_scores.sort_values(ascending=False).head(top_k)
    return list(top.items())


prod_faiss_index, prod_embeddings = build_or_load_dense_index(
    prod_chunks, cache_prefix="prod", force_rebuild=FORCE_REBUILD_PROD_INDEX
)
print(dense_search(prod_faiss_index, prod_chunks, "baltic dry index falls", top_k=5))

Loading cached embeddings/index for 'prod' from disk


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[(48, 0.8518450856208801), (10, 0.8480992317199707), (3, 0.847786545753479), (11, 0.8474568128585815), (45, 0.8472043871879578)]


## Stage 4 — Hybrid fusion (Reciprocal Rank Fusion)

BM25 scores and cosine-similarity scores live on **completely different,
incomparable scales** (BM25 is an unbounded term-frequency statistic; cosine
similarity is bounded in [-1, 1] and tends to cluster near 0.7-0.9 for
"somewhat relevant"). Adding or averaging them directly would let whichever
score happens to have larger magnitude dominate, for reasons that have
nothing to do with relevance.

**Reciprocal Rank Fusion (RRF)** sidesteps this by throwing away the raw
scores and fusing on *rank* instead:

    RRF(doc) = sum over each ranked list L containing doc of  1 / (k + rank_L(doc))

with `k = 60` (the standard default from the original RRF paper — it
flattens the curve so the difference between rank 1 and rank 2 matters less
than the difference between being ranked at all vs. not). A document that
appears near the top of *both* BM25 and dense rankings scores higher than one
that's great in only one list, which is exactly the "hybrid" behavior we
want.

In [10]:
def rrf_fuse(ranked_lists: list[list[tuple[int, float]]], k: int = 60, top_k: int = 100):
    """Fuse multiple (article_id, score) ranked lists via Reciprocal Rank Fusion.

    Only the *rank* within each list is used, not the raw score, which is why
    RRF is safe to apply across lists with incompatible score scales (BM25 vs
    cosine similarity).
    """
    rrf_scores: dict[int, float] = {}
    for ranked_list in ranked_lists:
        for rank, (article_id, _score) in enumerate(ranked_list, start=1):
            rrf_scores[article_id] = rrf_scores.get(article_id, 0.0) + 1.0 / (k + rank)

    fused = sorted(rrf_scores.items(), key=lambda kv: kv[1], reverse=True)
    return fused[:top_k]


def hybrid_search(
    bm25, chunks_df, faiss_index, query: str, bm25_k=200, dense_k=200, fused_k=100,
    recent: dict | None = None,
):
    """`recent`, if given (see `recent_window()`), restricts both BM25 and
    dense retrieval to chunks from articles published in that date window
    before they're ever fused -- the hard recency filter triggered by
    query wording like "recently"/"lately" (see the markdown cell above
    Stage 6b)."""
    allowed_mask = recent["mask"] if recent else None
    allowed_positions = recent["positions"] if recent else None
    bm25_hits = bm25_search(bm25, chunks_df, query, top_k=bm25_k, allowed_mask=allowed_mask)
    dense_hits = dense_search(faiss_index, chunks_df, query, top_k=dense_k, allowed_positions=allowed_positions)
    return rrf_fuse([bm25_hits, dense_hits], k=60, top_k=fused_k)


print(hybrid_search(prod_bm25, prod_chunks, prod_faiss_index, "baltic dry index falls", fused_k=5))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[(48, 0.032266458495966696), (11, 0.031754032258064516), (10, 0.0315136476426799), (3, 0.030798389007344232), (2, 0.030330882352941176)]


## Stage 5 — Cross-encoder reranking

A **bi-encoder** (what Stage 3 uses) embeds the query and each document
*independently* into vectors and compares them with a cheap dot product —
that's why it scales to searching 100k chunks in milliseconds. A
**cross-encoder** instead feeds the *(query, document)* pair **together**
through one transformer and outputs a single relevance score. It sees the
query and document interact token-by-token (full cross-attention), so it's
noticeably more accurate at fine-grained relevance judgments — but it's also
far more expensive: you can't precompute document vectors, every
(query, document) pair needs its own forward pass.

That cost/accuracy tradeoff is exactly why cross-encoders are used as a
**reranking** step on a *small* candidate set (here, the top 100 from RRF
fusion) rather than as the primary retriever over the whole corpus: running
`ms-marco-MiniLM-L-6-v2` on 100 pairs takes a fraction of a second; running it
on all 50k articles would take minutes per query.

We score on `(query, title + first ~300 words of the article)` — the opening
of a news article is where the lede/summary lives, so it's the highest-signal
slice to hand the cross-encoder.

In [11]:
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
_rerank_model = None


def get_rerank_model() -> CrossEncoder:
    global _rerank_model
    if _rerank_model is None:
        _rerank_model = CrossEncoder(RERANK_MODEL_NAME)
    return _rerank_model


def rerank_snippet(row: pd.Series, n_words: int = 300) -> str:
    title = str(row["title"])
    body_words = str(row["text"]).split()[:n_words]
    return title + ". " + " ".join(body_words)


def rerank(query: str, candidate_ids: list[int], df: pd.DataFrame) -> list[tuple[int, float]]:
    """Rerank `candidate_ids` with the cross-encoder, return
    (article_id, score) sorted descending."""
    model = get_rerank_model()
    cand_df = df.set_index("article_id").loc[candidate_ids]
    pairs = [(query, rerank_snippet(row)) for _, row in cand_df.iterrows()]
    scores = model.predict(pairs, show_progress_bar=False)
    reranked = sorted(zip(candidate_ids, scores), key=lambda kv: kv[1], reverse=True)
    return reranked


fused = hybrid_search(prod_bm25, prod_chunks, prod_faiss_index, "baltic dry index falls", fused_k=20)
reranked_demo = rerank("baltic dry index falls", [aid for aid, _ in fused][:20], articles)
print(reranked_demo[:5])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[(58, np.float32(9.557933)), (57, np.float32(9.44819)), (56, np.float32(9.382435)), (35, np.float32(9.381774)), (3, np.float32(9.381723))]


## Stage 6 — Recency boost (optional)

Shipping-market news ages fast: a six-month-old "rates outlook" piece is far
less useful than a similar one from this week, even if both are an equally
good *topical* match. We add a small, **optional and separately switchable**
boost so relevance stays primary and recency only breaks ties/nudges the
order — it's not a replacement for relevance ranking.

`final_score = relevance_score + recency_weight * recency_score`, where
`recency_score` linearly rescales each candidate's date into [0, 1] across
the date range *of the candidate set being boosted* (newest = 1, oldest = 0).
`recency_weight` defaults small (0.05) so it only matters when relevance
scores are close.

In [12]:
def recency_boost(
    scored_ids: list[tuple[int, float]], df: pd.DataFrame, weight: float = 0.05
) -> list[tuple[int, float]]:
    """Add a small linear recency bonus to each (article_id, score) pair.

    Disabled by passing weight=0.0 (or simply not calling this function) —
    kept as a separate, switchable step from the core relevance pipeline.
    """
    if weight == 0.0 or not scored_ids:
        return scored_ids

    ids = [aid for aid, _ in scored_ids]
    dates = df.set_index("article_id").loc[ids, "date_parsed"]
    date_ordinals = dates.map(lambda d: d.toordinal() if pd.notna(d) else np.nan)

    valid = date_ordinals.dropna()
    if valid.empty or valid.min() == valid.max():
        return scored_ids  # nothing to normalize against

    lo, hi = valid.min(), valid.max()
    recency_norm = ((date_ordinals - lo) / (hi - lo)).fillna(0.0)

    boosted = [
        (aid, score + weight * recency_norm.loc[aid])
        for aid, score in scored_ids
    ]
    return sorted(boosted, key=lambda kv: kv[1], reverse=True)

## Stage 6b — Query-triggered recency filter

The soft recency *boost* above (Stage 6) nudges already-retrieved results toward newer
articles, but it never stops an old, highly-relevant article from being retrieved in the
first place. That's wrong for a question like *"has the Baltic Dry Index been falling
recently?"* — the user means "right now," full stop, and a perfectly-matching article
from 2016 answering about a 2016 slump is not a correct answer to a 2026 question, no
matter how high it scores.

So when the query itself uses recency language ("recently", "lately", "latest", "this
month", "past few weeks", ...) **and doesn't name a different year**, we hard-filter the
candidate pool down to this calendar month's articles *before* BM25/dense retrieval ever
runs — not a score nudge, an exclusion. If this month doesn't have enough articles yet
(e.g. the scrape hasn't caught up), the window widens back one month at a time until it
clears `RECENT_MIN_ARTICLES`, and results say so rather than silently answering from a
different period than the user asked about.

This is deliberately separate from the Stage 6 soft boost: `recent_window()` answers "what
time period is the user actually asking about," independent of whether `use_recency`
(the soft score boost) is also switched on. Both `search_pipeline()` below (plain search)
and `rag_answer()` in Part 2 (RAG) call `is_recency_query()` on the raw query text and, if
it fires, pass the resulting window into retrieval automatically — this mirrors the same
feature in the Flask dashboard (`dry_bulk_rag_dashboard/rag_engine.py`), ported here for
parity.

In [13]:
# How many articles a date window needs before it's considered a usable
# "recent" answer; if a window has fewer, it's widened one month at a time.
RECENT_MIN_ARTICLES = 20

RECENCY_PATTERN = re.compile(
    r"\b(recent(ly)?|lately|latest|newest|current(ly)?|nowadays|these days|right now|"
    r"at the moment|at present|presently|this (week|month)|"
    r"(past|last) (few |couple of )?(days|weeks?|month))\b",
    re.IGNORECASE,
)
YEAR_PATTERN = re.compile(r"\b(19|20)\d{2}\b")
OTHER_YEAR_PATTERN = re.compile(r"\b(last|previous|past) years?\b|\byears? ago\b", re.IGNORECASE)


def is_recency_query(query: str, today: pd.Timestamp | None = None) -> bool:
    """True if `query` uses recency language ("recently", "lately", ...) and
    doesn't name a specific different year (or a vaguer "years ago" scale
    that recency language alone wouldn't correctly answer)."""
    if not RECENCY_PATTERN.search(query):
        return False
    year = (today or pd.Timestamp.today()).year
    asks_other_year = any(int(m.group()) != year for m in YEAR_PATTERN.finditer(query))
    return not (asks_other_year or OTHER_YEAR_PATTERN.search(query))


# Each chunk's article's publication date, in the same row order as
# `prod_chunks` -- lets us build a boolean mask / FAISS row-position array
# for "only chunks published in this date range" without a per-query join.
chunk_dates = (
    articles.set_index("article_id")["date_parsed"]
    .reindex(prod_chunks["article_id"])
    .to_numpy()
)


def recent_window(today: pd.Timestamp | None = None) -> dict:
    """Today's calendar month, widened back a month at a time if it has
    fewer than RECENT_MIN_ARTICLES articles (e.g. the scrape hasn't caught
    up with the current month yet)."""
    today = (today or pd.Timestamp.today()).normalize()
    month_start = today.replace(day=1)
    end = month_start + pd.offsets.MonthBegin(1)  # exclusive
    dates = articles["date_parsed"]
    start = month_start
    while ((dates >= start) & (dates < end)).sum() < RECENT_MIN_ARTICLES and start > dates.min():
        start -= pd.offsets.MonthBegin(1)

    requested = month_start.strftime("%B %Y")
    label = requested if start == month_start else f"{start.strftime('%B %Y')} to {requested}"
    mask = (chunk_dates >= start) & (chunk_dates < end)
    return {
        "start": start, "end": end, "label": label, "requested": requested,
        "fallback": start != month_start,
        "n_articles": int(((dates >= start) & (dates < end)).sum()),
        "mask": mask,                       # bool per chunk row, for BM25
        "positions": np.flatnonzero(mask),  # FAISS row positions
    }


print("Example recent window:", recent_window())
print("is_recency_query examples:",
      is_recency_query("Has the Baltic Dry Index been falling recently?"),
      is_recency_query("What happened to the Baltic Dry Index in 2016?"))

Example recent window: {'start': Timestamp('2026-09-01 00:00:00'), 'end': Timestamp('2026-11-01 00:00:00'), 'label': 'September 2026 to October 2026', 'requested': 'October 2026', 'fallback': True, 'n_articles': 111, 'mask': array([False, False, False, ..., False, False, False], shape=(92290,)), 'positions': array([  161,   162,   163,   164,   165,   166,   167,   168,   169,
         441,   442,   443,   444,   445,   446,   447,   448,   449,
         450,   451,   452,   453,   454,   455,   456,   457,   458,
       77974, 77975, 77976, 77977, 77978, 77979, 77980, 77981, 77982,
       77983, 77984, 77985, 77986, 77987, 77988, 77989, 77990, 77991,
       77992, 77993, 77994, 77995, 77996, 77997, 77998, 77999, 78000,
       78001, 78002, 78003, 78004, 78005, 78006, 78007, 78008, 78009,
       78010, 78011, 78012, 78013, 78014, 78015, 78016, 78017, 78018,
       78019, 78020, 78021, 78283, 78284, 78285, 78286, 78287, 78288,
       78289, 78290, 78291, 78292, 78293, 78294, 78295, 7829

In [14]:
def search_pipeline(
    query: str,
    df: pd.DataFrame,
    bm25: BM25Okapi,
    chunks_df: pd.DataFrame,
    faiss_index: faiss.Index,
    bm25_k: int = 200,
    dense_k: int = 200,
    fused_k: int = 100,
    use_rerank: bool = True,
    use_recency: bool = False,
    recency_weight: float = 0.05,
    recent: dict | None = None,
) -> list[tuple[int, float]]:
    """Run the full configurable pipeline: BM25 -> dense -> RRF fusion ->
    [optional rerank] -> [optional recency boost]. Returns (article_id, score)
    pairs sorted best-first. This is the single entry point every stage of
    the evaluation below (and the final search cell) calls with different
    flags switched on/off.

    `recent` is the hard, query-triggered recency filter from `recent_window()`
    (restricts the candidate pool to a date window before anything else runs);
    `use_recency`/`recency_weight` is the separate, always-optional soft
    recency *boost* from Stage 6 (nudges already-retrieved results by date).
    The two can be combined but answer different questions: `recent` says
    "only look at this time period at all," the soft boost says "given these
    candidates, slightly prefer the newer ones."
    """
    fused = hybrid_search(bm25, chunks_df, faiss_index, query, bm25_k, dense_k, fused_k, recent=recent)

    if use_rerank:
        candidate_ids = [aid for aid, _ in fused]
        results = rerank(query, candidate_ids, df)
    else:
        results = fused

    if use_recency:
        results = recency_boost(results, df, weight=recency_weight)

    return results

## Evaluation, part 1 — automatic "known-item" proxy

We don't have human relevance judgments for this corpus, so we build a cheap
**automatic proxy**: sample ~300 articles, use each article's **title** as
the query, and search over an index built from **body text only** (no
titles). The "correct" answer for each query is the one article the title
came from. If a stage's search is working, the body of that very article
should rank highly even though the title string never appears anywhere in
the index.

**Why body-only and not the production (title+body) index:** if titles were
in the index, a query like "Baltic Dry Index falls to 2729 down 42 points"
would trivially exact-match itself via BM25 on the title field — that would
tell us nothing about whether the retrieval actually understands content,
only that string matching works on identical strings.

**Limitation (important, and worth saying out loud in an interview):** this
is a *known-item* / self-retrieval proxy, not a relevance judgment. It tells
us "can the system find the one document a title was lifted from," which
rewards exact/near-exact phrase and entity overlap between title and body.
It does **not** tell us how well the system handles queries phrased
differently from the source text, multi-relevant-document queries, or truly
ambiguous queries — that's what the manual eval set (part 2) is for. Treat
these numbers as a regression check ("did a change make retrieval worse"),
not an absolute quality score.

Metrics, computed per stage (BM25 only / dense only / hybrid / hybrid+rerank):
- **Recall@10 / Recall@50** — is the true article anywhere in the top 10 / 50?
- **MRR** (Mean Reciprocal Rank) — `1 / rank_of_true_article`, averaged; 1.0
  is the true article is ranked #1 every time.
- **NDCG@10** — rank-sensitive quality score for a single relevant item,
  `1 / log2(rank + 1)` normalized, averaged.
- **Latency (ms)** — mean wall-clock time per query, so we can see the
  quality/speed tradeoff of adding reranking, not just the quality gain.

In [15]:
EVAL_SAMPLE_SIZE = 300

eval_rng = random.Random(SEED)
eval_sample_ids = eval_rng.sample(list(articles["article_id"]), k=min(EVAL_SAMPLE_SIZE, len(articles)))
eval_df = articles.set_index("article_id").loc[eval_sample_ids].reset_index()

# Body-only search field (deliberately excludes the title) for the eval index.
articles["body_only_text"] = articles["text"].fillna("")
eval_chunks = build_chunks(articles, "body_only_text")
print(f"Eval index: {len(articles)} articles -> {len(eval_chunks)} body-only chunks")

eval_bm25 = build_bm25_index(eval_chunks)
eval_faiss_index, eval_embeddings = build_or_load_dense_index(
    eval_chunks, cache_prefix="eval_body_only", force_rebuild=FORCE_REBUILD_EVAL_INDEX
)

Eval index: 48856 articles -> 90038 body-only chunks


Tokenizing for BM25:   0%|          | 0/90038 [00:00<?, ?it/s]

Embedding 90038 chunks for 'eval_body_only' (this is the slow, one-time step)


Batches:   0%|          | 0/1407 [00:00<?, ?it/s]

Cached embeddings to eval_body_only_embeddings.npy and index to eval_body_only_faiss.index


In [16]:
def recall_at_k(rank: float | None, k: int) -> float:
    return 1.0 if (rank is not None and rank <= k) else 0.0


def reciprocal_rank(rank: float | None) -> float:
    return 1.0 / rank if rank is not None else 0.0


def ndcg_at_k(rank: float | None, k: int) -> float:
    """NDCG@k for a single relevant document: DCG = 1/log2(rank+1) if the
    document is within the top k, else 0. IDCG (ideal DCG, relevant doc at
    rank 1) is 1/log2(2) = 1, so NDCG == DCG here."""
    if rank is not None and rank <= k:
        return 1.0 / np.log2(rank + 1)
    return 0.0


def find_rank(article_id: int, ranked_ids: list[int]) -> float | None:
    try:
        return ranked_ids.index(article_id) + 1
    except ValueError:
        return None


def run_eval_stage(stage_name: str, search_fn, eval_df: pd.DataFrame, k_recall=(10, 50), k_ndcg=10):
    """Run `search_fn(query) -> list[(article_id, score)]` for every title
    in `eval_df`, compare the rank of the true article, and return a metrics
    dict plus the mean latency in ms."""
    ranks = []
    latencies_ms = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"Evaluating: {stage_name}"):
        query = row["title"]
        true_id = row["article_id"]

        t0 = time.perf_counter()
        results = search_fn(query)
        latencies_ms.append((time.perf_counter() - t0) * 1000)

        ranked_ids = [aid for aid, _ in results]
        ranks.append(find_rank(true_id, ranked_ids))

    metrics = {
        "stage": stage_name,
        f"Recall@{k_recall[0]}": np.mean([recall_at_k(r, k_recall[0]) for r in ranks]),
        f"Recall@{k_recall[1]}": np.mean([recall_at_k(r, k_recall[1]) for r in ranks]),
        "MRR": np.mean([reciprocal_rank(r) for r in ranks]),
        f"NDCG@{k_ndcg}": np.mean([ndcg_at_k(r, k_ndcg) for r in ranks]),
        "avg_latency_ms": np.mean(latencies_ms),
    }
    return metrics

In [17]:
def eval_bm25_only(query):
    return bm25_search(eval_bm25, eval_chunks, query, top_k=200)


def eval_dense_only(query):
    return dense_search(eval_faiss_index, eval_chunks, query, top_k=200)


def eval_hybrid(query):
    return hybrid_search(eval_bm25, eval_chunks, eval_faiss_index, query, fused_k=100)


def eval_hybrid_rerank(query):
    fused = hybrid_search(eval_bm25, eval_chunks, eval_faiss_index, query, fused_k=100)
    candidate_ids = [aid for aid, _ in fused]
    return rerank(query, candidate_ids, articles)


eval_results = [
    run_eval_stage("BM25 only", eval_bm25_only, eval_df),
    run_eval_stage("Dense only", eval_dense_only, eval_df),
    run_eval_stage("Hybrid (RRF)", eval_hybrid, eval_df),
    run_eval_stage("Hybrid + Rerank", eval_hybrid_rerank, eval_df),
]

eval_comparison_table = pd.DataFrame(eval_results).set_index("stage")
eval_comparison_table

Evaluating: BM25 only:   0%|          | 0/300 [00:00<?, ?it/s]

Evaluating: Dense only:   0%|          | 0/300 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (895 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,Recall@10,Recall@50,MRR,NDCG@10,avg_latency_ms
stage,,,,,
BM25 only,0.896667,0.963333,0.730280,0.766990,213.743726
Dense only,0.873333,0.956667,0.752557,0.778526,81.019928
Hybrid (RRF),0.923333,0.986667,0.787975,0.818458,304.192078
Hybrid + Rerank,0.990000,0.990000,0.988333,0.988770,5238.902012


## Evaluation, part 2 — manual query set

The automatic eval above only measures self-retrieval. To judge *real*
search quality we need realistic queries a user would actually type, with
human-judged relevant documents. Below is a small, **editable** dict of ~15
shipping-market queries — fill in the `relevant_links` list for each query by
eyeballing the top-10 results (the helper below prints them for you) and
pasting in the `link` of every article you judge genuinely relevant. Once
filled in, this becomes a small but real relevance-judged eval set you can
rerun the same `run_eval_stage` logic over.

In [18]:
MANUAL_QUERIES: dict[str, list[str]] = {
    "Baltic Dry Index falls": [],
    "Baltic Dry Index climbs": [],
    "Maersk container orders": [],
    "port congestion China": [],
    "capesize rates outlook": [],
    "panamax freight rates": [],
    "newbuilding orders dry bulk": [],
    "iron ore exports Brazil": [],
    "coal imports China": [],
    "ship scrapping demolition": [],
    "Red Sea shipping disruption": [],
    "bunker fuel prices": [],
    "IMO emissions regulations": [],
    "dry bulk fleet supply": [],
    "grain exports freight demand": [],
}


def print_top_k(query: str, k: int = 10, **pipeline_kwargs) -> pd.DataFrame:
    """Run the full pipeline for `query` and pretty-print the top `k` results
    so you can manually judge relevance and fill in MANUAL_QUERIES. Applies
    the same query-triggered recency filter as run_search()/rag_answer()."""
    recent = pipeline_kwargs.pop("recent", None)
    if recent is None and is_recency_query(query):
        recent = recent_window()
    results = search_pipeline(
        query, articles, prod_bm25, prod_chunks, prod_faiss_index, recent=recent, **pipeline_kwargs
    )[:k]
    rows = []
    for rank, (aid, score) in enumerate(results, start=1):
        row = articles.set_index("article_id").loc[aid]
        rows.append(
            {
                "rank": rank,
                "score": round(float(score), 4),
                "title": row["title"],
                "date": row["date_parsed"],
                "source": row["source"],
                "link": row["link"],
            }
        )
    out = pd.DataFrame(rows)
    print(f"Query: {query!r}")
    return out


# Example — inspect results for the first manual query, then fill in
# MANUAL_QUERIES[...] with the links you judge relevant:
print_top_k(next(iter(MANUAL_QUERIES)))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: 'Baltic Dry Index falls'


,rank,score,title,date,source,link
0,1,9.5579,Baltic Dry Index falls to 2769 down 123 points,2021-11-04,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2769-down-123...
1,2,9.4482,Baltic Dry Index falls to 2807 down 37 points,2021-11-12,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2807-down-37-...
2,3,9.3824,Baltic Dry Index falls to 1233 down 07 points,2023-06-26,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-1233-down-07-...
3,4,9.3818,Baltic Dry Index falls to 2022 down 22 points,2025-08-18,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2022-down-22-...
4,5,9.3817,Baltic Dry Index falls to 3124 down 81 points,2026-06-03,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-3124-down-81-...
5,6,9.3759,Baltic Dry Index falls to 2017 down 14 points,2026-03-30,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2017-down-14-...
6,7,9.3595,Baltic Dry Index falls to 1674 down 15 points,2025-06-23,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-1674-down-15-...
7,8,9.3568,Baltic Dry Index falls to 1374 down 32 points,2025-05-07,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-1374-down-32-...
8,9,9.3389,Baltic Dry Index falls to 2729 down 42 points,2026-06-11,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2729-down-42-...
9,10,9.3234,Baltic Dry Index falls to 1598 down 04 points,2025-03-31,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-1598-down-04-...


In [19]:
def eval_manual_queries(manual_queries: dict[str, list[str]], search_fn, top_k: int = 50):
    """Evaluate Recall@10/50 and MRR against the hand-labeled relevant links.
    Queries with an empty relevant_links list are skipped (not yet labeled).
    """
    rows = []
    for query, relevant_links in manual_queries.items():
        if not relevant_links:
            continue
        results = search_fn(query)[:top_k]
        ranked_links = [articles.set_index("article_id").loc[aid, "link"] for aid, _ in results]
        hit_ranks = [ranked_links.index(l) + 1 for l in relevant_links if l in ranked_links]
        recall_10 = len([r for r in hit_ranks if r <= 10]) / len(relevant_links)
        recall_50 = len([r for r in hit_ranks if r <= 50]) / len(relevant_links)
        mrr = (1.0 / min(hit_ranks)) if hit_ranks else 0.0
        rows.append({"query": query, "Recall@10": recall_10, "Recall@50": recall_50, "MRR": mrr})

    if not rows:
        print("No manual queries labeled yet — fill in MANUAL_QUERIES[...] with relevant links first.")
        return pd.DataFrame(columns=["query", "Recall@10", "Recall@50", "MRR"])
    return pd.DataFrame(rows)


def manual_search_fn(query):
    return search_pipeline(query, articles, prod_bm25, prod_chunks, prod_faiss_index)


eval_manual_queries(MANUAL_QUERIES, manual_search_fn)

No manual queries labeled yet — fill in MANUAL_QUERIES[...] with relevant links first.


,query,Recall@10,Recall@50,MRR


## Final cell — the search interface

Edit `QUERY` (and optionally the date/source filters) and re-run this cell.
Everything else — chunking, BM25, dense retrieval, RRF fusion, reranking —
runs behind `search_pipeline()`.

In [20]:
# ---- Edit this line ---------------------------------------------------
QUERY = "baltic dry index drop"
# -------------------------------------------------------------------------

TOP_N = 50
DATE_FROM = None   # e.g. "2024-01-01" or None for no lower bound
DATE_TO = None     # e.g. "2026-12-31" or None for no upper bound
SOURCE_FILTER = None  # e.g. "splash", "Hellenic", "Breakwave", or None for all


def run_search(
    query: str,
    top_n: int = 50,
    date_from: str | None = None,
    date_to: str | None = None,
    source_filter: str | None = None,
    use_rerank: bool = True,
    use_recency: bool = False,
    recency_weight: float = 0.05,
) -> pd.DataFrame:
    # Query-triggered hard recency filter (see Stage 6b) -- "recently",
    # "lately", etc. restrict retrieval to the current/most-recent-available
    # month before anything else runs. DATE_FROM/DATE_TO above are a
    # separate, manual, always-available filter applied afterwards.
    recent = recent_window() if is_recency_query(query) else None
    if recent:
        note = " (widened, too few articles this month)" if recent["fallback"] else ""
        print(f"Recency language detected -> restricting to {recent['label']}{note}, "
              f"{recent['n_articles']} articles in window")

    # Over-fetch before filtering so date/source filters don't starve the
    # result set down below top_n.
    results = search_pipeline(
        query,
        articles,
        prod_bm25,
        prod_chunks,
        prod_faiss_index,
        fused_k=max(200, top_n * 4),
        use_rerank=use_rerank,
        use_recency=use_recency,
        recency_weight=recency_weight,
        recent=recent,
    )

    rows = []
    for aid, score in results:
        row = articles.set_index("article_id").loc[aid]
        if date_from and (pd.isna(row["date_parsed"]) or row["date_parsed"] < pd.Timestamp(date_from)):
            continue
        if date_to and (pd.isna(row["date_parsed"]) or row["date_parsed"] > pd.Timestamp(date_to)):
            continue
        if source_filter and row["source"] != source_filter:
            continue
        rows.append(
            {
                "rank": len(rows) + 1,
                "score": round(float(score), 4),
                "title": row["title"],
                "date": row["date_parsed"].date() if pd.notna(row["date_parsed"]) else None,
                "source": row["source"],
                "link": row["link"],
            }
        )
        if len(rows) >= top_n:
            break

    return pd.DataFrame(rows)


results_df = run_search(QUERY, top_n=TOP_N, date_from=DATE_FROM, date_to=DATE_TO, source_filter=SOURCE_FILTER)

try:
    from IPython.display import HTML, display

    display_df = results_df.copy()
    display_df["link"] = display_df["link"].map(lambda u: f'<a href="{u}" target="_blank">{u}</a>')
    display(HTML(display_df.to_html(escape=False, index=False)))
except ImportError:
    print(results_df.to_string(index=False))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

rank,score,title,date,source,link
1,9.4004,Baltic Dry Index falls to 2769 down 123 points,2021-11-04,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2769-down-123-points/
2,9.2970,Baltic Dry Index falls to 1233 down 07 points,2023-06-26,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-1233-down-07-points/
3,9.2599,Baltic Dry Index falls to 2729 down 42 points,2026-06-11,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2729-down-42-points/
4,9.2429,Baltic Dry Index falls to 3124 down 81 points,2026-06-03,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-3124-down-81-points/
5,9.2354,Baltic Dry Index falls to 2220 down 39 points,2025-09-29,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2220-down-39-points/
6,9.2254,Baltic Dry Index falls to 2807 down 37 points,2021-11-12,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2807-down-37-points/
7,9.2229,Baltic Dry Index falls to 2022 down 22 points,2025-08-18,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2022-down-22-points/
8,9.1980,Baltic Dry Index falls to 1674 down 15 points,2025-06-23,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-1674-down-15-points/
9,9.1672,Baltic Dry Index falls to 1374 down 32 points,2025-05-07,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-1374-down-32-points/
10,9.1271,Baltic Dry Index falls to 2017 down 14 points,2026-03-30,Hellenic,https://www.hellenicshippingnews.com/baltic-dry-index-falls-to-2017-down-14-points/


## Results & Learnings

The cell below renders the automatic evaluation table computed earlier in
this run straight into Markdown, so the numbers here are never stale/copied
by hand — re-running the notebook top to bottom regenerates them. The
free-text notes underneath are yours to fill in.

In [21]:
from IPython.display import Markdown, display

manual_eval_table = eval_manual_queries(MANUAL_QUERIES, manual_search_fn)
n_labeled = sum(1 for v in MANUAL_QUERIES.values() if v)

summary_md = f"""
### Automatic (known-item) evaluation — {len(eval_df)} sampled titles-as-queries

Body-only index, true document = the article the title came from. This is a
**self-retrieval proxy, not a human relevance judgment** — strong numbers
mean the pipeline reliably finds a document given text closely related to
its own content, not that it ranks real, differently-phrased user queries
well.

{eval_comparison_table.to_markdown()}

### Manual query-set evaluation — {n_labeled} / {len(MANUAL_QUERIES)} queries labeled

{manual_eval_table.to_markdown(index=False) if not manual_eval_table.empty else "*No queries labeled yet — fill in `MANUAL_QUERIES` with relevant links above and re-run.*"}

### Latency vs. quality

Compare `avg_latency_ms` in the automatic-eval table above across stages —
reranking is the expensive step (a full-attention forward pass per
candidate pair instead of a cached dot product), traded for whatever
Recall/MRR/NDCG gain it shows over hybrid-without-rerank.
"""

display(Markdown(summary_md))

No manual queries labeled yet — fill in MANUAL_QUERIES[...] with relevant links first.



### Automatic (known-item) evaluation — 300 sampled titles-as-queries

Body-only index, true document = the article the title came from. This is a
**self-retrieval proxy, not a human relevance judgment** — strong numbers
mean the pipeline reliably finds a document given text closely related to
its own content, not that it ranks real, differently-phrased user queries
well.

| stage           |   Recall@10 |   Recall@50 |      MRR |   NDCG@10 |   avg_latency_ms |
|:----------------|------------:|------------:|---------:|----------:|-----------------:|
| BM25 only       |    0.896667 |    0.963333 | 0.73028  |  0.76699  |         213.744  |
| Dense only      |    0.873333 |    0.956667 | 0.752557 |  0.778526 |          81.0199 |
| Hybrid (RRF)    |    0.923333 |    0.986667 | 0.787975 |  0.818458 |         304.192  |
| Hybrid + Rerank |    0.99     |    0.99     | 0.988333 |  0.98877  |        5238.9    |

### Manual query-set evaluation — 0 / 15 queries labeled

*No queries labeled yet — fill in `MANUAL_QUERIES` with relevant links above and re-run.*

### Latency vs. quality

Compare `avg_latency_ms` in the automatic-eval table above across stages —
reranking is the expensive step (a full-attention forward pass per
candidate pair instead of a cached dot product), traded for whatever
Recall/MRR/NDCG gain it shows over hybrid-without-rerank.


### My own observations *(fill in after reading the tables above)*

- Where did BM25 beat dense retrieval, and why (exact terms: ship names,
  numbers, index values)?
- Where did dense retrieval beat BM25 (paraphrases/synonyms it caught that
  keyword matching missed)?
- Did reranking change the *order* of the top 10 in ways that look like real
  quality improvements when reading the actual headlines, or mostly noise?
- Did the recency boost (if enabled) meaningfully change results for
  time-sensitive queries like "capesize rates outlook"?
- Which manual-set queries still came back poor, and what's missing in the
  pipeline for those cases (e.g. no query expansion, no handling of
  abbreviations like "BDI")?
- What would you change first if asked "how would you improve this" —
  candidates: query expansion/synonyms, a bigger embedding model, HNSW once
  the corpus grows past this scale, learning-to-rank instead of RRF,
  fine-tuning the reranker on this domain's negatives.

*(your notes here)*

# Part 2 — RAG: Answering Questions with a Local LLM

Everything above this point is **retrieval**: given a query, find the most relevant
articles. This part adds **generation** on top of it — a local LLM reads the retrieved
passages and writes a natural-language answer, citing which passage(s) it used.

This is RAG (Retrieval-Augmented Generation): instead of asking an LLM a question and
hoping it "knows" the answer from training data (which it might hallucinate, or might be
out of date — these are shipping-news articles, the LLM's training data isn't), we hand
it the actual retrieved text and tell it to answer **only** from that text.

Everything here runs **locally and for free** via [Ollama](https://ollama.com/), using
`qwen2.5:3b` through the `ollama` Python package — no OpenAI/Anthropic keys, no network
calls except to the local Ollama server on `localhost`.

In [22]:
# Additional install for this part (uncomment to run):
# !pip install -U ollama
# Also requires the Ollama server running locally (https://ollama.com/download).

import ollama

OLLAMA_MODEL = "qwen2.5:3b"

# How many chunks (not whole articles) get handed to the LLM as context.
RAG_TOP_K_CHUNKS = 5
# Candidate pool sizes for the chunk-level hybrid retrieval below -- smaller
# than the article-level search_pipeline's defaults since we only need a
# handful of individual passages, not a ranked list of 50k articles.
RAG_BM25_K = 100
RAG_DENSE_K = 100
RAG_FUSED_K = 30


def ensure_ollama_model(model_name: str) -> None:
    """Pull `model_name` via the local Ollama server if it isn't already
    present, printing download progress. One-time cost per machine."""
    local_models = {m.model for m in ollama.list().models}
    if model_name in local_models:
        print(f"Ollama model '{model_name}' already present.")
        return

    print(f"Pulling Ollama model '{model_name}' (one-time download)...")
    last_pct = -1
    for progress in ollama.pull(model_name, stream=True):
        if progress.total and progress.completed:
            pct = int(100 * progress.completed / progress.total)
            if pct != last_pct:
                print(f"\r{progress.status}: {pct}%", end="", flush=True)
                last_pct = pct
    print(f"\nDone pulling '{model_name}'.")


ensure_ollama_model(OLLAMA_MODEL)

Ollama model 'qwen2.5:3b' already present.


## Step 1 — Chunk-level hybrid retrieval

The `search_pipeline()` built earlier returns whole **articles**, because that's what a
human searcher wants to click into. An LLM answering a question doesn't need a whole
article — it needs the specific passage that actually contains the answer, and feeding it
a long article it has to read past wastes context and makes citations vague ("the article
says..." vs. "passage [2] says...").

So RAG retrieval here reuses every building block from Part 1 — the same BM25 index, the
same FAISS index, the same cross-encoder, the same Reciprocal Rank Fusion function — but
skips the "max-aggregate chunk scores up to the article" step. BM25 and dense search each
return their top candidate **chunks** directly, RRF fuses those chunk-level rankings, and
the cross-encoder reranks the fused chunk candidates on the chunk text itself (not a
300-word article snippet). The top 5 survivors become the LLM's context.

In [23]:
def bm25_search_chunks_topk(
    bm25: BM25Okapi, chunks_df: pd.DataFrame, query: str, top_k: int = 100,
    allowed_mask: np.ndarray | None = None,
):
    """Like `bm25_search`, but returns top_k (chunk_id, score) pairs directly
    -- no max-aggregation up to the article level."""
    scores = bm25_search_chunks(bm25, chunks_df, query)
    chunk_ids = chunks_df["chunk_id"].values
    order = np.argsort(-scores.values)
    if allowed_mask is not None:
        order = order[allowed_mask[order]]
    order = order[:top_k]
    return [(int(chunk_ids[i]), float(scores.values[i])) for i in order]


def dense_search_chunks_topk(
    index: faiss.Index, chunks_df: pd.DataFrame, query: str, top_k: int = 100,
    allowed_positions: np.ndarray | None = None,
):
    """Like `dense_search`, but returns top_k (chunk_id, score) pairs
    directly from the FAISS index -- no aggregation."""
    query_vec = embed_texts([query], is_query=True)
    scores, positions = _faiss_search(index, query_vec, top_k, allowed_positions)
    chunk_ids = chunks_df.iloc[positions]["chunk_id"].values
    return [(int(cid), float(s)) for cid, s in zip(chunk_ids, scores)]


def retrieve_top_chunks(
    query: str,
    bm25: BM25Okapi,
    chunks_df: pd.DataFrame,
    faiss_index: faiss.Index,
    df: pd.DataFrame,
    bm25_k: int = RAG_BM25_K,
    dense_k: int = RAG_DENSE_K,
    fused_k: int = RAG_FUSED_K,
    final_k: int = RAG_TOP_K_CHUNKS,
    recent: dict | None = None,
) -> list[dict]:
    """Chunk-level hybrid retrieval + cross-encoder rerank, reusing the same
    BM25 index, FAISS index, RRF fusion (`rrf_fuse`), and cross-encoder model
    (`get_rerank_model`) as the article-level search_pipeline. Returns the
    final_k best chunks as dicts with their source article's metadata
    attached, ready to be labeled [1]..[n] for the LLM prompt.

    `recent` (see Stage 6b's `recent_window()`) restricts retrieval to a
    date window before fusion, same as `search_pipeline()` in Part 1 -- so a
    "what's happening recently" question can't be answered from a chunk
    published years ago just because it scores well.
    """
    allowed_mask = recent["mask"] if recent else None
    allowed_positions = recent["positions"] if recent else None
    bm25_hits = bm25_search_chunks_topk(bm25, chunks_df, query, top_k=bm25_k, allowed_mask=allowed_mask)
    dense_hits = dense_search_chunks_topk(faiss_index, chunks_df, query, top_k=dense_k, allowed_positions=allowed_positions)
    fused = rrf_fuse([bm25_hits, dense_hits], k=60, top_k=fused_k)

    model = get_rerank_model()
    chunk_ids = [cid for cid, _ in fused]
    cand = chunks_df.set_index("chunk_id").loc[chunk_ids]
    pairs = [(query, text) for text in cand["chunk_text"]]
    scores = model.predict(pairs, show_progress_bar=False)
    reranked = sorted(zip(chunk_ids, scores), key=lambda kv: kv[1], reverse=True)[:final_k]

    results = []
    for cid, score in reranked:
        chunk_row = chunks_df.set_index("chunk_id").loc[cid]
        article = df.set_index("article_id").loc[chunk_row["article_id"]]
        results.append(
            {
                "chunk_id": int(cid),
                "score": float(score),
                "article_id": int(chunk_row["article_id"]),
                "chunk_text": chunk_row["chunk_text"],
                "title": article["title"],
                "date": article["date_parsed"],
                "source": article["source"],
                "link": article["link"],
            }
        )
    return results


_demo_chunks = retrieve_top_chunks("baltic dry index falls", prod_bm25, prod_chunks, prod_faiss_index, articles)
for c in _demo_chunks:
    print(f"[{c['chunk_id']}] score={c['score']:.3f}  {c['title'][:60]}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[158] score=9.558  Baltic Dry Index falls to 2769 down 123 points
[157] score=9.448  Baltic Dry Index falls to 2807 down 37 points
[156] score=9.382  Baltic Dry Index falls to 1233 down 07 points
[93] score=9.382  Baltic Dry Index falls to 2022 down 22 points
[4] score=9.382  Baltic Dry Index falls to 3124 down 81 points


## Step 2 — Grounded answer generation (why this reduces hallucination)

LLMs hallucinate because, by default, nothing stops them from generating a plausible-
sounding fact that isn't actually true — they're predicting likely next tokens, not
looking anything up. RAG's whole premise is to change the task from "recall a fact from
memory" to "read this text and summarize/extract from it," which is a task LLMs are much
more reliable at. Three concrete things in the prompt below do the work:

1. **"Answer using only information in the context. Do not use prior knowledge."** —
   explicitly tells the model not to fall back on its training data, which is both
   possibly wrong and definitely not sourced.
2. **Mandatory citations (`[n]`)** — forcing every claim to point at a specific passage
   makes fabrication visible. If the model writes a claim with no `[n]`, or cites a `[n]`
   that doesn't exist, that's a detectable failure (see the citation-validity check
   below) rather than a silent one.
3. **An explicit, exact refusal phrase for insufficient context** — without this
   instruction, a model asked something the context doesn't cover will often still try to
   answer (filling the gap with prior knowledge, i.e. hallucinating) rather than admit it
   doesn't know.

None of this makes hallucination impossible — a small 3B model can still misread a
passage, or cite a passage that doesn't actually support what it wrote (which is exactly
why the faithfulness check in the evaluation section below exists). It makes hallucination
*detectable and far less likely*, which is the realistic bar for a local, free system.

In [24]:
RAG_REFUSAL_PHRASE = "The articles don't contain enough information to answer this question."

RAG_SYSTEM_PROMPT = f"""You are a research assistant answering questions about dry bulk shipping news using ONLY the numbered context passages provided below.

Rules:
- Answer using only information in the context. Do not use prior knowledge.
- Every factual claim must be followed by a citation like [1] or [2] referencing the passage it came from.
- If the context does not contain enough information to answer, respond exactly: "{RAG_REFUSAL_PHRASE}"
- Each passage header shows its publication date. When passages disagree, prefer the most recent one.
- Be concise."""


def build_context_block(top_chunks: list[dict]) -> str:
    """Format retrieved chunks as [1]..[n] labeled passages, each with its
    source article's title, source, and date, so citations are traceable
    back to a real article without the LLM needing to repeat the metadata."""
    blocks = []
    for i, c in enumerate(top_chunks, start=1):
        date_str = c["date"].date() if pd.notna(c["date"]) else "unknown date"
        header = f"[{i}] {c['title']} ({c['source']}, {date_str})"
        blocks.append(f"{header}\n{c['chunk_text']}")
    return "\n\n".join(blocks)


def generate_answer(
    question: str, context_block: str, model: str = OLLAMA_MODEL, recent: dict | None = None
) -> tuple[str, float]:
    """Call the local Ollama model with the grounded system prompt + context,
    return (answer_text, generation_latency_ms).

    `recent`, if given, adds an explicit note pinning "recent"/"lately" in
    the question to the actual date window retrieval was restricted to --
    without it, the model has no way to know what "recently" should mean
    relative to today, since its own training data has a different cutoff.
    """
    note = ""
    if recent is not None:
        note = (
            f"\n\nNote: this question is about recent developments. 'Recent' means "
            f"{recent['label']}, and all passages above were published in that period. "
            f"Talk only about that period."
        )
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context_block}\n\nQuestion: {question}{note}"},
    ]
    t0 = time.perf_counter()
    response = ollama.chat(model=model, messages=messages)
    latency_ms = (time.perf_counter() - t0) * 1000
    return response["message"]["content"], latency_ms


_demo_context = build_context_block(_demo_chunks)
_demo_answer, _demo_ms = generate_answer("Did the Baltic Dry Index fall recently?", _demo_context)
print(_demo_answer)
print(f"\n(generation: {_demo_ms:.0f} ms)")

Yes, the most recent fall mentioned is 81 points to 3124 on June 3, 2026 [5].

(generation: 32617 ms)


In [25]:
def rag_answer(
    question: str,
    bm25: BM25Okapi = None,
    chunks_df: pd.DataFrame = None,
    faiss_index: faiss.Index = None,
    df: pd.DataFrame = None,
    final_k: int = RAG_TOP_K_CHUNKS,
    model: str = OLLAMA_MODEL,
) -> dict:
    """End-to-end RAG: retrieve the top final_k chunks for `question`, build
    the labeled context block, generate a grounded answer, and time both
    stages separately. Defaults to the production indexes built in Part 1.

    If `question` uses recency language ("recently", "lately", ...), this
    automatically restricts retrieval to the current/most-recent-available
    month (Stage 6b) and tells the LLM what that window is -- the same
    behavior as the Flask dashboard's `Engine.answer()`.
    """
    bm25 = bm25 if bm25 is not None else prod_bm25
    chunks_df = chunks_df if chunks_df is not None else prod_chunks
    faiss_index = faiss_index if faiss_index is not None else prod_faiss_index
    df = df if df is not None else articles

    recent = recent_window() if is_recency_query(question) else None

    t0 = time.perf_counter()
    top_chunks = retrieve_top_chunks(question, bm25, chunks_df, faiss_index, df, final_k=final_k, recent=recent)
    retrieval_ms = (time.perf_counter() - t0) * 1000

    context_block = build_context_block(top_chunks)
    answer, generation_ms = generate_answer(question, context_block, model=model, recent=recent)

    return {
        "question": question,
        "recent": {k: v for k, v in recent.items() if k not in ("mask", "positions")} if recent else None,
        "top_chunks": top_chunks,
        "context_block": context_block,
        "answer": answer,
        "retrieval_ms": retrieval_ms,
        "generation_ms": generation_ms,
    }


def extract_citations(answer_text: str) -> list[int]:
    """Parse all [n] citation markers out of an answer's text."""
    return sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer_text)})


_demo_result = rag_answer("Did the Baltic Dry Index fall recently?")
print(_demo_result["answer"])
print("Cited:", extract_citations(_demo_result["answer"]))
print("Recency window used:", _demo_result["recent"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

The articles don't contain enough information to answer this question.
Cited: []
Recency window used: {'start': Timestamp('2026-09-01 00:00:00'), 'end': Timestamp('2026-11-01 00:00:00'), 'label': 'September 2026 to October 2026', 'requested': 'October 2026', 'fallback': True, 'n_articles': 111}


## Final cell — ask a question

Edit `QUESTION` and re-run this cell. It prints the grounded answer, the sources it
actually cited (with links back to the source articles), and how much time went to
retrieval vs. generation.

In [26]:
# ---- Edit this line ---------------------------------------------------
QUESTION = "Has the Baltic Dry Index been falling recently?"
# -------------------------------------------------------------------------

result = rag_answer(QUESTION)
cited = extract_citations(result["answer"])

print(f"Q: {result['question']}\n")

if result["recent"]:
    r = result["recent"]
    note = " (widened -- too few articles in the requested month)" if r["fallback"] else ""
    print(f"[Recency language detected -> answered from {r['label']}{note}, "
          f"{r['n_articles']} articles in window]\n")

print(f"A: {result['answer']}\n")

print("Cited sources:")
if cited:
    for n in cited:
        if 1 <= n <= len(result["top_chunks"]):
            c = result["top_chunks"][n - 1]
            date_str = c["date"].date() if pd.notna(c["date"]) else "unknown date"
            print(f"  [{n}] {c['title']} — {c['source']}, {date_str}\n       {c['link']}")
        else:
            print(f"  [{n}] *** cited but does not exist in the retrieved context ***")
else:
    print("  (none)")

print(f"\nretrieval: {result['retrieval_ms']:.0f} ms | generation: {result['generation_ms']:.0f} ms")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q: Has the Baltic Dry Index been falling recently?

[Recency language detected -> answered from September 2026 to October 2026 (widened -- too few articles in the requested month), 111 articles in window]

A: The articles don't contain enough information to answer this question.

Cited sources:
  (none)

retrieval: 1725 ms | generation: 27417 ms


## RAG Evaluation

Three different things can go wrong with a RAG answer, so we check each separately:

1. **Context recall** — did retrieval even surface a relevant article in the top 5
   chunks? If not, no amount of good generation can save the answer; this is a retrieval
   failure, not a generation failure. Checked against a hand-filled `expected_links` list
   per question (you fill these in by hand, same as the manual query set in Part 1).
2. **Faithfulness** — given that the right context *was* retrieved, did the LLM's answer
   actually stick to it, or did it add something not supported by any passage? We can't
   hand-check every answer, so we use the **same local LLM as an automatic judge**: a
   second prompt hands it the context and the answer and asks "is every claim here
   supported by the context?" This is the standard "LLM-as-judge" pattern, and it has a
   real limitation worth stating plainly: a model judging its own output tends to be
   lenient on its own style of mistakes (self-preference bias), and a 3B model is not a
   highly reliable judge to begin with. Treat this as a cheap, directional signal — "did
   faithfulness get worse after I changed the prompt" — not a certified accuracy score.
3. **Citation validity** — do the `[n]` markers in the answer actually correspond to
   passages that exist in the context handed to the model (n between 1 and however many
   chunks were retrieved)? This is cheap to check exactly (no LLM needed) and catches a
   distinct failure mode: the model citing a passage number that doesn't exist, or that
   exists but doesn't support the claim next to it (validity checks the citation
   *exists*, not that it's *correct* — faithfulness is what checks correctness).
4. **Refusal correctness** — 3 of the 15 questions below are deliberately outside this
   corpus (crypto prices, sports, geography). A well-behaved system should retrieve weak,
   irrelevant context for these and then refuse rather than answer anyway. We check this
   by looking for the exact refusal phrase in the output.

In [27]:
# Editable: fill in `expected_links` by hand (same workflow as MANUAL_QUERIES in
# Part 1 -- use rag_answer(question)["top_chunks"] to see what's retrieved, then
# paste in links you judge genuinely relevant). The last 3 are deliberately
# unanswerable from this corpus, to test refusal.
RAG_EVAL_QUESTIONS: list[dict] = [
    {"question": "Has the Baltic Dry Index been falling recently?", "expected_links": [], "expect_refusal": False},
    {"question": "What is driving demand for capesize vessels?", "expected_links": [], "expect_refusal": False},
    {"question": "How is Chinese iron ore demand affecting dry bulk rates?", "expected_links": [], "expect_refusal": False},
    {"question": "What is happening with newbuilding orders for dry bulk ships?", "expected_links": [], "expect_refusal": False},
    {"question": "Why are panamax freight rates under pressure?", "expected_links": [], "expect_refusal": False},
    {"question": "What is the outlook for coal shipments to China?", "expected_links": [], "expect_refusal": False},
    {"question": "How has the Red Sea disruption affected shipping routes?", "expected_links": [], "expect_refusal": False},
    {"question": "What is happening with ship scrapping and demolition activity?", "expected_links": [], "expect_refusal": False},
    {"question": "How are IMO emissions regulations affecting the shipping industry?", "expected_links": [], "expect_refusal": False},
    {"question": "What is driving grain export demand this year?", "expected_links": [], "expect_refusal": False},
    {"question": "How are bunker fuel prices trending?", "expected_links": [], "expect_refusal": False},
    {"question": "What is the state of the dry bulk newbuilding fleet supply pipeline?", "expected_links": [], "expect_refusal": False},
    {"question": "What is the current price of Bitcoin?", "expected_links": [], "expect_refusal": True},
    {"question": "Who won the most recent Super Bowl?", "expected_links": [], "expect_refusal": True},
    {"question": "What is the capital of Australia?", "expected_links": [], "expect_refusal": True},
]


def citation_validity(citations: list[int], n_context_chunks: int) -> bool | None:
    """None if the answer made no citations (nothing to check); otherwise
    True only if every cited [n] is within the retrieved context's range."""
    if not citations:
        return None
    return all(1 <= n <= n_context_chunks for n in citations)


def check_refusal(answer_text: str) -> bool:
    return RAG_REFUSAL_PHRASE.lower() in answer_text.lower()


def context_recall_hit(expected_links: list[str], top_chunks: list[dict]) -> bool | None:
    """None if no expected links were labeled yet for this question."""
    if not expected_links:
        return None
    retrieved_links = {c["link"] for c in top_chunks}
    return any(link in retrieved_links for link in expected_links)


JUDGE_PROMPT_TEMPLATE = """You are a strict fact-checking judge. Given a CONTEXT and an ANSWER that was supposed to be derived only from that context, decide if every claim in the ANSWER is supported by the CONTEXT.

CONTEXT:
{context}

ANSWER:
{answer}

Respond with exactly one word, YES or NO, on the first line. YES means every claim in the answer is supported by the context. NO means at least one claim is not supported or is fabricated."""


def judge_faithfulness(context_block: str, answer: str, model: str = OLLAMA_MODEL) -> tuple[bool, str, float]:
    """Use the same local LLM, in a separate call with a separate prompt, to
    judge whether `answer` is fully supported by `context_block`."""
    t0 = time.perf_counter()
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": JUDGE_PROMPT_TEMPLATE.format(context=context_block, answer=answer)}],
    )
    latency_ms = (time.perf_counter() - t0) * 1000
    verdict_text = response["message"]["content"].strip()
    supported = verdict_text.upper().startswith("YES")
    return supported, verdict_text, latency_ms

In [28]:
def run_rag_eval(questions: list[dict]) -> pd.DataFrame:
    """Run the full RAG pipeline + all four checks for every question,
    return one row per question."""
    rows = []
    for item in tqdm(questions, desc="RAG eval"):
        r = rag_answer(item["question"])
        citations = extract_citations(r["answer"])
        rows.append(
            {
                "question": item["question"],
                "expect_refusal": item["expect_refusal"],
                "refused": check_refusal(r["answer"]),
                "context_recall": context_recall_hit(item["expected_links"], r["top_chunks"]),
                "citations_valid": citation_validity(citations, len(r["top_chunks"])),
                "faithful": judge_faithfulness(r["context_block"], r["answer"])[0],
                "retrieval_ms": r["retrieval_ms"],
                "generation_ms": r["generation_ms"],
            }
        )
    return pd.DataFrame(rows)


rag_eval_df = run_rag_eval(RAG_EVAL_QUESTIONS)
rag_eval_df

RAG eval:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (7 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,question,expect_refusal,refused,context_recall,citations_valid,faithful,retrieval_ms,generation_ms
0,Has the Baltic Dry Index been falling recently?,False,True,None,None,False,1773.1362,4.651806e+02
1,What is driving demand for capesize vessels?,False,False,None,None,False,3128.6510,2.829483e+04
2,How is Chinese iron ore demand affecting dry bulk rates?,False,False,None,None,True,1528.0009,3.027834e+04
3,What is happening with newbuilding orders for dry bulk ships?,False,False,None,None,False,1412.9297,2.992841e+04
4,Why are panamax freight rates under pressure?,False,False,None,None,False,1354.1379,3.912425e+07
5,What is the outlook for coal shipments to China?,False,False,None,None,False,3049.1255,6.821646e+03
6,How has the Red Sea disruption affected shipping routes?,False,False,None,None,True,1644.2078,6.437393e+03
7,What is happening with ship scrapping and demolition activity?,False,False,None,None,False,1418.8397,9.894147e+03
8,How are IMO emissions regulations affecting the shipping industry?,False,False,None,None,True,1729.2310,8.058456e+03
9,What is driving grain export demand this year?,False,False,None,None,False,1648.3586,5.498164e+03


In [29]:
answerable = rag_eval_df[~rag_eval_df["expect_refusal"]]
refusal_rows = rag_eval_df[rag_eval_df["expect_refusal"]]
labeled_recall = answerable["context_recall"].dropna()
labeled_citations = rag_eval_df["citations_valid"].dropna()

rag_eval_summary = pd.Series(
    {
        "context_recall (labeled Qs only)": labeled_recall.mean() if len(labeled_recall) else float("nan"),
        "n_labeled_for_recall": len(labeled_recall),
        "faithfulness_rate (answerable Qs)": answerable["faithful"].mean(),
        "citation_validity_rate": labeled_citations.mean() if len(labeled_citations) else float("nan"),
        "refusal_accuracy (unanswerable Qs)": refusal_rows["refused"].mean(),
        "false_refusal_rate (answerable Qs)": answerable["refused"].mean(),
        "avg_retrieval_ms": rag_eval_df["retrieval_ms"].mean(),
        "avg_generation_ms": rag_eval_df["generation_ms"].mean(),
    },
    name="value",
)
rag_eval_summary.to_frame()

,value
context_recall (labeled Qs only),NaN
n_labeled_for_recall,0.000000e+00
faithfulness_rate (answerable Qs),3.333333e-01
citation_validity_rate,1.000000e+00
refusal_accuracy (unanswerable Qs),1.000000e+00
false_refusal_rate (answerable Qs),8.333333e-02
avg_retrieval_ms,1.747129e+03
avg_generation_ms,2.618690e+06


## Design choices & limitations

**Why a small (3B) local model instead of a bigger one.** `qwen2.5:3b` is small enough to
generate an answer in a few seconds on a laptop CPU with no GPU — that's the whole point
of "local and free." The tradeoff is real: a 3B model reasons less reliably than a
70B-class model or a hosted frontier model, is more easily confused by subtly contradictory
passages, and is a noticeably weaker judge (see below). If this were a product rather than
a demo/interview project, the natural next step is a bigger local model (Ollama's 8B-13B
tier) or a hosted API for generation only, while keeping retrieval exactly as-is — the
retrieval pipeline doesn't care what generates the final answer.

**Why LLM-as-judge, and why to distrust it a little.** Hand-labeling faithfulness for
every answer doesn't scale, so an LLM judging another LLM's output is the standard cheap
substitute. The catch: using the *same* model as both answerer and judge means any
systematic blind spot (e.g. consistently missing a particular kind of unsupported
inference) shows up in neither the answer nor the check — the judge shares the same
model's failure modes. A more rigorous setup would judge with a different, stronger model,
or have a human spot-check a sample of the judge's verdicts to calibrate how much to trust
it. As built, treat the faithfulness rate as a trend indicator across changes to this
notebook, not an absolute faithfulness percentage.

**Chunk-size trade-offs.** The same ~200-word / 50-word-overlap chunks from Part 1 are
reused here as RAG context. Smaller chunks give the LLM a tighter, more focused passage
per citation (good — less for it to misread or cherry-pick from) but risk splitting a
fact across a chunk boundary so no single passage fully contains it. Larger chunks give
more surrounding context per citation but dilute the embedding (a long chunk's vector
represents an average of several ideas, which can hurt retrieval precision) and give the
LLM more text to potentially misattribute a claim to. 200/50 is a reasonable middle
ground for ~1,500-character news articles; it wasn't re-tuned specifically for the RAG
use case here, which a more careful system would do separately from the retrieval-only
tuning in Part 1.

**Citation validity is necessary, not sufficient.** A citation existing (pointing at a
real passage) doesn't mean that passage actually supports the specific claim next to it —
the model could cite passage [2] for a claim [2] doesn't contain. That gap is exactly why
citation validity and faithfulness are two separate checks above, not one.

**Context recall depends on manual labeling.** Like the manual query set in Part 1, the
`expected_links` here start empty — the recall number only becomes meaningful once you've
filled some in by hand, and 15 questions is a small, non-random sample. It's enough to
sanity-check the pipeline and to catch regressions, not enough to make a statistically
confident claim about retrieval quality.